# 07 — Identity candidate evaluation and versioned OSM snapshots

This notebook uses **fictional records**. It demonstrates source-version checks and measures the candidate-review rule against explicit pair labels. The evaluation is a regression test, not a real-world matching-accuracy estimate.

## 1. Load and inspect reference labels

A review *candidate* is not an automatically accepted identity match. Each pair label is independently specified in the benchmark, including examples where the blocking heuristic is wrong.

In [ ]:
from pathlib import Path
from savoroute.evaluation.identity import load_identity_benchmark, evaluate_identity_candidates

benchmark = load_identity_benchmark(Path("data/sample/identity_pairs_fixture.json"))
print(benchmark.name, "records:", len(benchmark.records), "annotated pairs:", len(benchmark.pairs))
assert benchmark.synthetic and len(benchmark.pairs) == 14

## 2. Evaluate the rule used by OSM synchronization

The rule proposes review when normalized names match and either addresses match or coordinates are within 75 m. It does not infer business identity.

In [ ]:
summary = evaluate_identity_candidates(benchmark)
print(summary.as_dict())
assert summary.true_positive == 1
assert summary.false_positive == 3
assert summary.false_negative == 1
assert summary.true_negative == 9

## 3. Inspect disagreements

The false-positive examples share names and proximity but have different reference identities. The false-negative example has a name variation. This is why review candidates are not automatically merged.

In [ ]:
for label, pairs in (("false positive", summary.false_positive_pairs),
                     ("false negative", summary.false_negative_pairs)):
    for left, right in pairs:
        print(label, left, benchmark.records[left].name, "|", right, benchmark.records[right].name)

## 4. Bootstrap a version-aware source registry

The next fixture has fictional OSM-style versions and UTC timestamps. The importer retains them through the report and registry.

In [ ]:
import json
from savoroute.ingestion.osm import import_osm_snapshot
from savoroute.identity.registry import bootstrap_osm_registry
from savoroute.services.osm_sync import sync_osm_snapshot

original = json.loads(Path("data/sample/osm_overpass_fixture_versioned.json").read_text())
import_result = import_osm_snapshot(original, synthetic_fixture=True)
registry = bootstrap_osm_registry(import_result.catalog, import_result.report())
key = ("osm", "node/900000001")
print("Version:", registry.lookup(*key).source_version)
assert registry.lookup(*key).source_version == 1

## 5. Accept a newer snapshot, reject an older one

Version 2 changes a catalog field. Replaying version 1 afterward must not reverse the update.

In [ ]:
new_data = json.loads(Path("data/sample/osm_overpass_update_fixture_versioned.json").read_text())
new_data["elements"] = [new_data["elements"][0]]
updated = sync_osm_snapshot(new_data, base_catalog=import_result.catalog,
                            registry=registry, synthetic_fixture=True)
print(updated.decisions[0].as_dict())
assert updated.decisions[0].status == "updated"
assert updated.registry.lookup(*key).source_version == 2

old_data = {"savoroute_fixture": True, "elements": [original["elements"][0]]}
stale = sync_osm_snapshot(old_data, base_catalog=updated.catalog,
                          registry=updated.registry, synthetic_fixture=True)
print(stale.decisions[0].as_dict())
assert stale.decisions[0].status == "review"
assert stale.catalog == updated.catalog

## 6. Prepare an annotation worksheet for a bounded snapshot

The worksheet contains **null** labels. It must be manually reviewed before evaluation. For real OSM data, retain source attribution and check ODbL obligations before publishing a derived dataset.

In [ ]:
from savoroute.identity.annotations import prepare_osm_annotations
from savoroute.ingestion.sources import IngestionError
from savoroute.evaluation.identity import parse_identity_benchmark

worksheet = prepare_osm_annotations(original, max_records=6, max_pairs=8)
print("Unlabeled pairs:", len(worksheet["pairs"]))
assert all(pair["same_entity"] is None for pair in worksheet["pairs"])
try:
    parse_identity_benchmark(worksheet)
except IngestionError as exc:
    print("Evaluation correctly blocked:", exc)
else:
    raise AssertionError("Unreviewed pairs must not be evaluated")

## Interpretation

- **Version ordering** protects previously accepted, versioned records; historical registries without version metadata have no rollback baseline until their first versioned sync.
- **Precision, recall, and review rate** are conditional on the annotated pair sample. The sampling procedure is not representative of all restaurant pairs.
- **Real-data validation remains pending.** It requires an independently reviewed, sufficiently diverse sample from permitted source data and reporting of excluded/ambiguous pairs.